# MLP ReLU Experiment: Baseline ReLU vs Dynamic ReLU

This notebook compares MLP architectures using ReLU activations:

1. **Baseline (ReLU)**: Standard fixed ReLU hidden layers + Softmax output
2. **Dynamic ReLU (Per-Neuron)**: Each neuron has learnable `a, b` parameters

**Training Approach**:
- First train weights with baseline ReLU
- Then copy model, replace with Dynamic ReLU, and finetune activation parameters only (weights frozen)

**Dynamic ReLU Function**: `max(a, b * x)` where:
- `a` controls the floor / leak threshold
- `b` controls the positive slope
- When a=0, b=1, reduces to standard ReLU

**Gradients** (mathematically correct backprop):
- $\frac{\partial f}{\partial a} = \mathbb{1}[b \cdot x \leq a]$ → used to update $a$
- $\frac{\partial f}{\partial b} = x \cdot \mathbb{1}[b \cdot x > a]$ → used to update $b$

**Datasets**: MNIST, Fashion-MNIST, CIFAR-10 (20 seeds each)

In [1]:
import sys
import os

# Add src to path for imports
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Tuple
from dataclasses import dataclass, field
from scipy import stats

from data_utils import DatasetConfig, DataManager
from mlp_trainer import MLPTrainer, ExperimentResult
from mlp import MLP, MLPConfig, create_baseline_mlp

print("Imports successful!")

Imports successful!


## Configuration

In [2]:
# Experiment Configuration
N_SEEDS = 20
SEEDS = [42, 123, 456, 789, 1001, 1111, 2222, 3333, 4444, 5555,
         6666, 7777, 8888, 9999, 1010, 2020, 3030, 4040, 5050, 6060]
HIDDEN_DIMS = [256, 128]  # Two hidden layers
EPOCHS = 30
BATCH_SIZE = 128
LEARNING_RATE = 0.01
ACTIVATION_EPOCHS = 30

print("Configuration:")
print(f"  Number of seeds: {N_SEEDS}")
print(f"  Hidden layers: {HIDDEN_DIMS}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Activation training epochs: {ACTIVATION_EPOCHS}")

Configuration:
  Number of seeds: 20
  Hidden layers: [256, 128]
  Epochs: 30
  Batch size: 128
  Learning rate: 0.01
  Activation training epochs: 30


## Helper Classes and Functions

In [3]:
@dataclass
class AggregatedResult:
    """Aggregated results from multiple seeds."""
    model_name: str
    train_accs: List[float] = field(default_factory=list)
    test_accs: List[float] = field(default_factory=list)
    times: List[float] = field(default_factory=list)
    epochs: List[int] = field(default_factory=list)

    @property
    def train_mean(self) -> float:
        return np.mean(self.train_accs)

    @property
    def train_std(self) -> float:
        return np.std(self.train_accs)

    @property
    def test_mean(self) -> float:
        return np.mean(self.test_accs)

    @property
    def test_std(self) -> float:
        return np.std(self.test_accs)

    @property
    def time_mean(self) -> float:
        return np.mean(self.times)

    @property
    def epochs_mean(self) -> float:
        return np.mean(self.epochs)

In [4]:
def load_dataset(dataset_type: str, test_size: float = 0.2, random_state: int = 42):
    """Load and prepare dataset."""
    print(f"Loading {dataset_type} dataset...")
    config = DatasetConfig(
        dataset_type=dataset_type,
        test_size=test_size,
        random_state=random_state,
    )
    dm = DataManager(config)
    X_train, X_test, y_train, y_test = dm.generate_dataset()

    print(f"  Training samples: {len(X_train):,}")
    print(f"  Test samples: {len(X_test):,}")
    print(f"  Input features: {X_train.shape[1]}")
    print(f"  Classes: {len(np.unique(y_train))}")

    return X_train, X_test, y_train, y_test

In [5]:
def print_results(agg_results: Dict[str, AggregatedResult], dataset_name: str, n_seeds: int) -> None:
    """Print comparison of aggregated experiment results."""
    print("\n" + "=" * 100)
    print(f"EXPERIMENT RESULTS: {dataset_name.upper()} ({n_seeds} seeds)")
    print("=" * 100)

    headers = ["Model", "Train Acc", "Test Acc", "Time (s)", "Epochs"]
    row_format = "{:<45} {:>18} {:>18} {:>10} {:>8}"

    print(row_format.format(*headers))
    print("-" * 100)

    for name, r in agg_results.items():
        print(row_format.format(
            name[:45],
            f"{r.train_mean:.4f} \u00b1 {r.train_std:.4f}",
            f"{r.test_mean:.4f} \u00b1 {r.test_std:.4f}",
            f"{r.time_mean:.2f}",
            f"{r.epochs_mean:.1f}",
        ))

    print("-" * 100)

    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    baseline = agg_results.get("Baseline (ReLU)", list(agg_results.values())[0])
    improvement = best.test_mean - baseline.test_mean

    print(f"\n\U0001f3c6 Best: {best_name}")
    print(f"   Test Accuracy: {best.test_mean:.4f} \u00b1 {best.test_std:.4f}")
    print(f"   Improvement over baseline: {improvement:+.4f} ({improvement*100:+.2f}%)")

In [6]:
def save_results(all_results: dict, filepath: str, n_seeds: int) -> None:
    """Save aggregated results to CSV file."""
    rows = []
    for dataset_name, agg_results in all_results.items():
        for model_name, r in agg_results.items():
            rows.append({
                "dataset": dataset_name,
                "model_name": model_name,
                "n_seeds": n_seeds,
                "train_acc_mean": r.train_mean,
                "train_acc_std": r.train_std,
                "test_acc_mean": r.test_mean,
                "test_acc_std": r.test_std,
                "time_mean_seconds": r.time_mean,
                "epochs_mean": r.epochs_mean,
                "train_accs": str(r.train_accs),
                "test_accs": str(r.test_accs),
                "timestamp": datetime.now().isoformat(),
            })

    df = pd.DataFrame(rows)
    df.to_csv(filepath, index=False)
    print(f"\nResults saved to: {filepath}")

## ReLU Experiment Functions

In [7]:
def run_relu_baseline(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Tuple[MLP, ExperimentResult]:
    """
    Train baseline ReLU MLP.

    Returns both the trained model and the experiment result.
    """
    np.random.seed(seed)

    input_dim = X_train.shape[1]
    output_dim = len(np.unique(y_train))

    model = create_baseline_mlp(
        input_dim=input_dim,
        hidden_dims=hidden_dims,
        output_dim=output_dim,
        learning_rate=learning_rate,
    )

    if verbose:
        print("\n" + "=" * 60)
        print("BASELINE MLP (ReLU)")
        print("=" * 60)
        print(model.summary())

    trainer = MLPTrainer(
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )

    history = trainer.train(model, X_train, y_train, X_test, y_test)

    train_acc = trainer._compute_accuracy(model, X_train, y_train)
    test_acc = trainer._compute_accuracy(model, X_test, y_test)

    result = ExperimentResult(
        model_name="Baseline (ReLU)",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=history.training_time,
        epochs_trained=history.epochs_trained,
        model_params=model.num_params,
    )

    return model, result

In [8]:
def run_dynamic_relu_finetuning(
    baseline_model: MLP,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    per_neuron: bool = True,
    verbose: bool = False,
) -> ExperimentResult:
    """
    Take a trained baseline ReLU model, copy it with DynamicReLU activations,
    and train only the activation parameters (weights frozen).

    DynamicReLU: f(x) = max(a, b * x)
    Initialized with a=0, b=1 → identical to standard ReLU at start.
    """
    dynamic_model = baseline_model.copy_with_dynamic_activations(
        per_neuron=per_neuron,
        activation_lr=learning_rate,
    )

    mode_str = "Per-Neuron" if per_neuron else "Per-Layer"

    if verbose:
        print("\n" + "=" * 60)
        print(f"DYNAMIC RELU FINETUNING ({mode_str})")
        print("=" * 60)
        print("Starting from trained baseline ReLU weights...")
        print(dynamic_model.summary())

    trainer = MLPTrainer(
        epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        verbose=verbose,
    )

    pre_train_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)
    if verbose:
        print(f"\nTest accuracy before activation training: {pre_train_acc:.4f}")
        print(f"\n--- Training activation parameters (weights frozen) ---")

    activation_history = trainer.train_activation_params(
        dynamic_model, X_train, y_train,
        epochs=activation_epochs,
        X_val=X_test, y_val=y_test,
    )

    train_acc = trainer._compute_accuracy(dynamic_model, X_train, y_train)
    test_acc = trainer._compute_accuracy(dynamic_model, X_test, y_test)

    if verbose:
        print(f"\nActivation training complete.")
        print(f"Test accuracy: {test_acc:.4f} (was {pre_train_acc:.4f}, improvement: {test_acc - pre_train_acc:+.4f})")

    activation_info = []
    for layer in dynamic_model.layers:
        if hasattr(layer, 'activation') and layer.activation.is_learnable:
            activation_info.append(layer.activation.info)

    return ExperimentResult(
        model_name=f"Dynamic ReLU Finetuning ({mode_str})",
        train_accuracy=train_acc,
        test_accuracy=test_acc,
        training_time=activation_history.training_time,
        epochs_trained=activation_history.epochs_trained,
        model_params=dynamic_model.num_params,
        activation_info="; ".join(activation_info),
    )

In [9]:
def run_single_seed_relu(
    X_train: np.ndarray,
    X_test: np.ndarray,
    y_train: np.ndarray,
    y_test: np.ndarray,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    verbose: bool = False,
) -> Dict[str, ExperimentResult]:
    """Run all ReLU experiment variants for a single seed."""
    results = {}

    # 1. Train Baseline ReLU MLP
    baseline_model, baseline_result = run_relu_baseline(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        hidden_dims=hidden_dims,
        epochs=epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        seed=seed,
        verbose=verbose,
    )
    results["Baseline (ReLU)"] = baseline_result

    # 2. Copy baseline and finetune with DynamicReLU (Per-Neuron)
    activation_result = run_dynamic_relu_finetuning(
        baseline_model=baseline_model,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        activation_epochs=activation_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        per_neuron=True,
        verbose=verbose,
    )
    results["Dynamic ReLU Finetuning (Per-Neuron)"] = activation_result

    return results

In [10]:
def run_relu_experiment_suite(
    dataset_type: str,
    hidden_dims: List[int],
    epochs: int,
    activation_epochs: int,
    batch_size: int,
    learning_rate: float,
    seeds: List[int],
) -> Dict[str, AggregatedResult]:
    """Run ReLU experiments with multiple seeds and aggregate results."""

    # Load data once (use first seed for train/test split)
    X_train, X_test, y_train, y_test = load_dataset(dataset_type, random_state=seeds[0])

    model_names = [
        "Baseline (ReLU)",
        "Dynamic ReLU Finetuning (Per-Neuron)",
    ]
    agg_results = {name: AggregatedResult(name) for name in model_names}

    for i, seed in enumerate(seeds):
        print(f"\n  Seed {i+1}/{len(seeds)} (seed={seed})...")

        seed_results = run_single_seed_relu(
            X_train, X_test, y_train, y_test,
            hidden_dims=hidden_dims,
            epochs=epochs,
            activation_epochs=activation_epochs,
            batch_size=batch_size,
            learning_rate=learning_rate,
            seed=seed,
            verbose=False,
        )

        for name, result in seed_results.items():
            agg_results[name].train_accs.append(result.train_accuracy)
            agg_results[name].test_accs.append(result.test_accuracy)
            agg_results[name].times.append(result.training_time)
            agg_results[name].epochs.append(result.epochs_trained)

        baseline_acc = seed_results["Baseline (ReLU)"].test_accuracy
        best_acc = max(r.test_accuracy for r in seed_results.values())
        print(f"    Baseline: {baseline_acc:.4f}, Best: {best_acc:.4f}")

    return agg_results

## Run Experiments

In [11]:
print("=" * 100)
print("MLP RELU EXPERIMENT: Baseline ReLU vs Dynamic ReLU Activation Finetuning")
print("Approach: Train baseline ReLU once, copy model, finetune activation params on copy")
print("=" * 100)

all_results = {}

MLP RELU EXPERIMENT: Baseline ReLU vs Dynamic ReLU Activation Finetuning
Approach: Train baseline ReLU once, copy model, finetune activation params on copy


### MNIST Dataset

In [12]:
print("\n" + "\U0001f31f" * 40)
print("DATASET: MNIST (Handwritten Digits)")
print("\U0001f31f" * 40)

mnist_results = run_relu_experiment_suite(
    dataset_type='mnist',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['mnist'] = mnist_results
print_results(mnist_results, "MNIST", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: MNIST (Handwritten Digits)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.9629, Best: 0.9634

  Seed 2/20 (seed=123)...
    Baseline: 0.9635, Best: 0.9635

  Seed 3/20 (seed=456)...
    Baseline: 0.9641, Best: 0.9641

  Seed 4/20 (seed=789)...
    Baseline: 0.9651, Best: 0.9651

  Seed 5/20 (seed=1001)...
    Baseline: 0.9642, Best: 0.9642

  Seed 6/20 (seed=1111)...
    Baseline: 0.9640, Best: 0.9640

  Seed 7/20 (seed=2222)...
    Baseline: 0.9641, Best: 0.9642

  Seed 8/20 (seed=3333)...
    Baseline: 0.9646, Best: 0.9649

  Seed 9/20 (seed=4444)...
    Baseline: 0.9639, Best: 0.9639

  Seed 10/20 (seed=5555)...
    Baseline: 0.9631, Best: 0.9631

  Seed 11/20 (seed=6666)...
    Baseline: 0.9647, Best: 0.9650

  Seed 12/20 (seed=7777)...
    Baseline: 0.9638, Best: 0.9639

  Seed 13/20 (s

### Fashion-MNIST Dataset

In [13]:
print("\n" + "\U0001f31f" * 40)
print("DATASET: FASHION-MNIST")
print("\U0001f31f" * 40)

fashion_results = run_relu_experiment_suite(
    dataset_type='fashion_mnist',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['fashion_mnist'] = fashion_results
print_results(fashion_results, "Fashion-MNIST", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: FASHION-MNIST
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading fashion_mnist dataset...
  Training samples: 56,000
  Test samples: 14,000
  Input features: 784
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.8876, Best: 0.8881

  Seed 2/20 (seed=123)...
    Baseline: 0.8849, Best: 0.8854

  Seed 3/20 (seed=456)...
    Baseline: 0.8869, Best: 0.8869

  Seed 4/20 (seed=789)...
    Baseline: 0.8859, Best: 0.8862

  Seed 5/20 (seed=1001)...
    Baseline: 0.8839, Best: 0.8860

  Seed 6/20 (seed=1111)...
    Baseline: 0.8821, Best: 0.8855

  Seed 7/20 (seed=2222)...
    Baseline: 0.8834, Best: 0.8854

  Seed 8/20 (seed=3333)...
    Baseline: 0.8837, Best: 0.8859

  Seed 9/20 (seed=4444)...
    Baseline: 0.8874, Best: 0.8881

  Seed 10/20 (seed=5555)...
    Baseline: 0.8864, Best: 0.8873

  Seed 11/20 (seed=6666)...
    Baseline: 0.8814, Best: 0.8826

  Seed 12/20 (seed=7777)...
    Baseline: 0.8851, Best: 0.8861

  Seed 13/20 (seed=8

### CIFAR-10 Dataset

In [14]:
print("\n" + "\U0001f31f" * 40)
print("DATASET: CIFAR-10 (Color Images)")
print("\U0001f31f" * 40)

cifar_results = run_relu_experiment_suite(
    dataset_type='cifar10',
    hidden_dims=HIDDEN_DIMS,
    epochs=EPOCHS,
    activation_epochs=ACTIVATION_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seeds=SEEDS,
)
all_results['cifar10'] = cifar_results
print_results(cifar_results, "CIFAR-10", N_SEEDS)


🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
DATASET: CIFAR-10 (Color Images)
🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟🌟
Loading cifar10 dataset...
CIFAR-10 loaded: 60000 samples, 3072 features
  Training samples: 48,000
  Test samples: 12,000
  Input features: 3072
  Classes: 10

  Seed 1/20 (seed=42)...
    Baseline: 0.5140, Best: 0.5210

  Seed 2/20 (seed=123)...
    Baseline: 0.5056, Best: 0.5138

  Seed 3/20 (seed=456)...
    Baseline: 0.5128, Best: 0.5134

  Seed 4/20 (seed=789)...
    Baseline: 0.5122, Best: 0.5198

  Seed 5/20 (seed=1001)...
    Baseline: 0.5148, Best: 0.5207

  Seed 6/20 (seed=1111)...
    Baseline: 0.5150, Best: 0.5189

  Seed 7/20 (seed=2222)...
    Baseline: 0.5144, Best: 0.5186

  Seed 8/20 (seed=3333)...
    Baseline: 0.5152, Best: 0.5199

  Seed 9/20 (seed=4444)...
    Baseline: 0.5131, Best: 0.5173

  Seed 10/20 (seed=5555)...
    Baseline: 0.5154, Best: 0.5190

  Seed 11/20 (seed=6666)...
    Baseline: 0.5180, Best: 0.5189

  Seed 12/20 (seed=7777)...
    B

## Statistical Significance

In [15]:
print("\n" + "=" * 100)
print("STATISTICAL SIGNIFICANCE (paired t-test: Baseline ReLU vs Dynamic ReLU)")
print("=" * 100)

for dataset_name, agg_results in all_results.items():
    baseline = agg_results["Baseline (ReLU)"]
    dynamic = agg_results["Dynamic ReLU Finetuning (Per-Neuron)"]

    t_stat, p_value = stats.ttest_rel(dynamic.test_accs, baseline.test_accs)
    delta = dynamic.test_mean - baseline.test_mean
    sig = "*" if p_value < 0.05 else ""

    print(f"\n  {dataset_name.upper()}:")
    print(f"    Baseline:     {baseline.test_mean:.4f} \u00b1 {baseline.test_std:.4f}")
    print(f"    Dynamic ReLU: {dynamic.test_mean:.4f} \u00b1 {dynamic.test_std:.4f}")
    print(f"    Delta:        {delta:+.4f} ({delta*100:+.2f}%)")
    print(f"    t={t_stat:.3f}, p={p_value:.4f} {sig}")


STATISTICAL SIGNIFICANCE (paired t-test: Baseline ReLU vs Dynamic ReLU)

  MNIST:
    Baseline:     0.9639 ± 0.0006
    Dynamic ReLU: 0.9639 ± 0.0007
    Delta:        -0.0000 (-0.00%)
    t=-0.206, p=0.8386 

  FASHION_MNIST:
    Baseline:     0.8845 ± 0.0020
    Dynamic ReLU: 0.8858 ± 0.0016
    Delta:        +0.0013 (+0.13%)
    t=5.472, p=0.0000 *

  CIFAR10:
    Baseline:     0.5155 ± 0.0038
    Dynamic ReLU: 0.5201 ± 0.0030
    Delta:        +0.0046 (+0.46%)
    t=7.942, p=0.0000 *


## Final Summary

In [16]:
print("\n" + "=" * 100)
print(f"FINAL SUMMARY (Mean \u00b1 Std over {N_SEEDS} seeds)")
print("=" * 100)

for dataset_name, agg_results in all_results.items():
    print(f"\n\U0001f4ca {dataset_name.upper()}:")

    baseline = agg_results["Baseline (ReLU)"]
    best_name = max(agg_results.keys(), key=lambda x: agg_results[x].test_mean)
    best = agg_results[best_name]
    improvement = best.test_mean - baseline.test_mean

    print(f"   Baseline (ReLU): {baseline.test_mean:.4f} \u00b1 {baseline.test_std:.4f}")
    print(f"   Best ({best_name}): {best.test_mean:.4f} \u00b1 {best.test_std:.4f}")
    print(f"   Improvement: {improvement:+.4f} ({improvement*100:+.2f}%)")


FINAL SUMMARY (Mean ± Std over 20 seeds)

📊 MNIST:
   Baseline (ReLU): 0.9639 ± 0.0006
   Best (Baseline (ReLU)): 0.9639 ± 0.0006
   Improvement: +0.0000 (+0.00%)

📊 FASHION_MNIST:
   Baseline (ReLU): 0.8845 ± 0.0020
   Best (Dynamic ReLU Finetuning (Per-Neuron)): 0.8858 ± 0.0016
   Improvement: +0.0013 (+0.13%)

📊 CIFAR10:
   Baseline (ReLU): 0.5155 ± 0.0038
   Best (Dynamic ReLU Finetuning (Per-Neuron)): 0.5201 ± 0.0030
   Improvement: +0.0046 (+0.46%)


## Save Results

In [17]:
output_path = "mlp_relu_experiment_results.csv"
save_results(all_results, output_path, N_SEEDS)

df = pd.read_csv(output_path)
df


Results saved to: mlp_relu_experiment_results.csv


,dataset,model_name,n_seeds,train_acc_mean,train_acc_std,test_acc_mean,test_acc_std,time_mean_seconds,epochs_mean,train_accs,test_accs,timestamp
0,mnist,Baseline (ReLU),20,0.989604,0.000440,0.963893,0.000643,15.104617,30.00,"[0.9890892857142857, 0.9906071428571429, 0.989...","[0.9629285714285715, 0.9635, 0.964142857142857...",2026-05-06T19:40:18.632126
1,mnist,Dynamic ReLU Finetuning (Per-Neuron),20,0.989621,0.000463,0.963882,0.000704,6.117154,10.25,"[0.9889285714285714, 0.990625, 0.9896607142857...","[0.9634285714285714, 0.9635, 0.964142857142857...",2026-05-06T19:40:18.632202
2,fashion_mnist,Baseline (ReLU),20,0.932426,0.001236,0.884504,0.002035,15.193847,30.00,"[0.9314821428571428, 0.9343392857142857, 0.933...","[0.8875714285714286, 0.8848571428571429, 0.886...",2026-05-06T19:40:18.632246
3,fashion_mnist,Dynamic ReLU Finetuning (Per-Neuron),20,0.934580,0.001376,0.885804,0.001564,7.019966,11.80,"[0.9335714285714286, 0.9357678571428572, 0.935...","[0.8881428571428571, 0.8853571428571428, 0.886...",2026-05-06T19:40:18.632285
4,cifar10,Baseline (ReLU),20,0.756653,0.010262,0.515500,0.003815,31.813154,21.60,"[0.75025, 0.7530208333333334, 0.76427083333333...","[0.514, 0.5055833333333334, 0.51275, 0.51225, ...",2026-05-06T19:40:18.632322
5,cifar10,Dynamic ReLU Finetuning (Per-Neuron),20,0.767435,0.009663,0.520092,0.003028,12.071073,10.05,"[0.7628958333333333, 0.7625625, 0.773895833333...","[0.521, 0.51375, 0.5134166666666666, 0.5198333...",2026-05-06T19:40:18.632358
